# Trabalho para G2

**Tema** · Preços de Combustíveis no Brasil

Gabriel de Moraes Ferreira Campos · Sistemas de Informação, 6º período  
Professor Alexandre Neves Louzada  
Linguagens de programação · Análise e Visualização de Dados com Python

Exploração dos três CSV de `archive.zip`. O notebook usa as mesmas rotinas do artigo, com verificações dos totais de entrada e cálculos por pergunta de pesquisa.


In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "data" / "raw").is_dir():
    root = root.parent
if not (root / "data" / "raw").is_dir():
    raise FileNotFoundError("Abra o notebook na pasta do projeto ou em notebooks/.")
sys.path.insert(0, str(root))

import pandas as pd
from src import etl, analise

raw = root / "data" / "raw"
processed = root / "data" / "processed"
panel, quality = etl.run(raw, processed)

inventory = pd.DataFrame(quality).T[
    ["linhas_brutas", "linhas_validas", "linhas_quarentena", "meses_observados"]
]
print(inventory.to_string())


       linhas_brutas linhas_validas linhas_quarentena meses_observados
Brasil           258            258                 0              258
Região          1290           1290                 0              258
UF              6966           6950                16              258


## Qualidade e cobertura

A lacuna global de setembro de 2020 é preservada. A série estadual tem 16 linhas sem mês e UF. Essas linhas são registradas individualmente no arquivo de quarentena. A tabela conta preços médios disponíveis por produto e escala, após validar chaves e tipos.


In [2]:
assert len(panel) == 59_486
assert not panel.duplicated(["referencia", "nivel", "localidade", "combustivel"]).any()
assert all(info["meses_ausentes"] == ["2020-09"] for info in quality.values())
assert quality["UF"]["linhas_quarentena"] == 16

coverage = panel.pivot_table(
    index="combustivel", columns="nivel", values="preco_medio", aggfunc="count"
)
coverage.index = coverage.index.map(lambda key: etl.PRODUCTS[key][0])
print(coverage.to_string())
print("\nAmapá · meses adicionais ausentes", len(quality["UF"]["meses_ausentes_por_uf"]["AP"]))


nivel                 Brasil  Região    UF
combustivel                               
Etanol hidratado         258    1290  6950
GLP P13                  258    1290  6950
Gás natural veicular     258    1215  4393
Gasolina aditivada        28     140   752
Gasolina comum           258    1290  6950
Óleo diesel              258    1290  6947
Óleo diesel S10          121     600  3235

Amapá · meses adicionais ausentes 16


## RQ1 · Evolução da gasolina comum no Brasil

Alinhar a série ao calendário permite manter setembro de 2020 em branco. As variações usam meses consecutivos. Os limites de Tukey sinalizam extremos para auditoria, e a correlação de Pearson é calculada entre variações de gasolina e etanol nos mesmos meses.


In [3]:
national = analise.serie_nacional(panel)
changes = analise.variacoes_nacionais(national)
diagnostic = analise.diagnostico_variacoes(changes)

gasoline = national[analise.GASOLINA]
peak = gasoline.idxmax()
print(f"Pico nacional · {peak:%m/%Y} · R$ {gasoline.loc[peak]:.2f}/L")
print(f"Dezembro de 2022 · R$ {gasoline.loc['2022-12'].iloc[0]:.2f}/L")
print(f"Variações com marcação IQR · {diagnostic['fora_dos_limites_iqr'].sum()} de {len(diagnostic)}")
print(f"Correlação de Pearson · {changes.corr().iloc[0, 1]:.3f}")
print()
print(diagnostic.nsmallest(4, "gasolina_var_pct")[["gasolina_var_pct", "fora_dos_limites_iqr"]].round(2).to_string())


Pico nacional · 05/2022 · R$ 7.28/L
Dezembro de 2022 · R$ 4.97/L
Variações com marcação IQR · 39 de 256
Correlação de Pearson · 0.668

            gasolina_var_pct  fora_dos_limites_iqr
referencia                                        
2022-07-01            -16.55                  True
2022-08-01            -10.91                  True
2002-01-01            -10.55                  True
2020-04-01             -8.87                  True


## RQ2 · Preço anual da gasolina por território

A média anual atribui o mesmo peso aos 12 meses. O ano de 2022 é o último com 12 meses na série nacional e inclui todas as UFs e regiões. Os agregados regionais vêm diretamente da tabela de regiões.


In [4]:
pairs = analise.comparacoes(panel)
annual = analise.medias_anuais(pairs)
year = int(annual.loc[annual["nivel"].eq("Brasil"), "ano"].max())

regional = annual.loc[annual["ano"].eq(year) & annual["nivel"].eq("Região")]
states = annual.loc[annual["ano"].eq(year) & annual["nivel"].eq("UF")]
assert len(regional) == 5 and len(states) == 27
prices = states["preco_medio_anual"]

print(f"Ano de comparação · {year}")
print(regional[["localidade", "preco_medio_anual", "meses"]].round(3).to_string(index=False))
print()
print(states.iloc[[0, -1]][["localidade", "preco_medio_anual"]].round(3).to_string(index=False))
print(f"Amplitude entre UFs · R$ {prices.max() - prices.min():.3f}/L")
print(f"Mediana · {prices.median():.3f} | Desvio-padrão amostral · {prices.std():.3f}")


Ano de comparação · 2022
  localidade  preco_medio_anual  meses
    Nordeste              6.254     12
       Norte              6.189     12
Centro-Oeste               6.11     12
     Sudeste              6.062     12
         Sul              6.017     12

localidade  preco_medio_anual
     Piauí              6.517
     Amapá              5.485
Amplitude entre UFs · R$ 1.032/L
Mediana · 6.180 | Desvio-padrão amostral · 0.226


## RQ3 · Razão entre os preços do etanol e da gasolina

A razão usa os dois preços médios da mesma localidade e do mesmo mês. O limiar de 70% serve como referência descritiva. O custo por quilômetro depende do consumo do veículo.


In [5]:
ratios = analise.frequencia_limiar(pairs)
ratio_year = ratios.loc[ratios["ano"].eq(year)].copy()
ratio_year["razao_media_pct"] = ratio_year["razao_media"] * 100
columns = ["localidade", "meses_validos", "meses_ate_70_pct", "razao_media_pct"]

print(ratio_year.loc[ratio_year["nivel"].ne("UF"), columns].round(2).to_string(index=False))
print()
print(ratio_year.loc[
    ratio_year["nivel"].eq("UF") & ratio_year["meses_ate_70_pct"].gt(0), columns
].round(2).to_string(index=False))


  localidade  meses_validos  meses_ate_70_pct  razao_media_pct
      Brasil             12                 3            72.61
Centro-Oeste             12                 5            70.44
    Nordeste             12                 0            79.84
       Norte             12                 0            83.44
     Sudeste             12                 3            71.88
         Sul             12                 0            79.42

  localidade  meses_validos  meses_ate_70_pct  razao_media_pct
       Goiás             12                 7            70.25
 Mato Grosso             12                 9            67.88
Minas Gerais             12                 2            72.91
     Paraíba             12                 1             77.9
   São Paulo             12                 4            71.93


## Verificação e publicação

Os módulos em `src/` geram as mesmas tabelas, os indicadores e o HTML interativo. Execute `python -m src.artigo` a partir da raiz para atualizar `docs/index.html`. As unidades, os dados ausentes e as fórmulas são descritos em `data/data_dictionary.md`.
